In [35]:
import numpy as np
import os
import time
from pydrake.all import (
    Meshcat,
    Simulator,
    RigidTransform,
    RollPitchYaw,
    InverseKinematics,
    Solve,
	RotationMatrix,
    SceneGraphCollisionChecker,
    MinimumDistanceLowerBoundConstraint,
    Context,
	BsplineTrajectory,
    TrajectorySource,
	PiecewisePolynomial,
	BasicVector,
	LeafSystem,
	DiagramBuilder,
)
from pydrake.multibody.plant import MultibodyPlant
from pydrake.planning import KinematicTrajectoryOptimization
from manipulation.station import LoadScenario, MakeHardwareStation
from pydrake.geometry import Sphere, Rgba
from manipulation.meshcat_utils import PublishPositionTrajectory

In [2]:
meshcat = Meshcat()

INFO:drake:Meshcat listening for connections at http://localhost:7001


In [141]:
# Set up paths
scenario_file = os.path.abspath("../kitchen_model/real_kitchen_scenario.yaml")
kitchen_model_path = os.path.abspath("../kitchen_model")
assets_path = os.path.abspath("../assets")

# Read the scenario YAML and replace placeholders
with open(scenario_file, 'r') as f:
    scenario_data = f.read()

scenario_data = scenario_data.replace("{KITCHEN_MODEL_PATH}", kitchen_model_path)
scenario_data = scenario_data.replace("{ASSETS_PATH}", assets_path)

# Load the scenario
scenario = LoadScenario(data=scenario_data)
station = MakeHardwareStation(scenario, meshcat)
sim = Simulator(station)
context = sim.get_mutable_context()

# Initialize robot to home position
sim.AdvanceTo(3.0)
print("Real kitchen scenario loaded successfully")

Real kitchen scenario loaded successfully


In [142]:
plant = station.GetSubsystemByName("plant")
plant_context = plant.GetMyMutableContextFromRoot(context)

# Get mobile base indices
ix = plant.GetJointByName("iiwa_base_x").position_start()
iy = plant.GetJointByName("iiwa_base_y").position_start()
iz = plant.GetJointByName("iiwa_base_z").position_start()
j1 = plant.GetJointByName("iiwa_joint_1").position_start()

# Get gripper frame
gripper_body = plant.GetBodyByName("body")

# Get initial configuration and limits
active_lo, active_hi = 3, 10
q0 = plant.GetPositions(plant_context).copy()
qlo = plant.GetPositionLowerLimits()
qhi = plant.GetPositionUpperLimits()
vlo = plant.GetVelocityLowerLimits()
vhi = plant.GetVelocityUpperLimits()

print(f"Initial configuration: {q0[3:10]}")
print(f"Initial (x,y,z) position: ({q0[ix]:.2f}, {q0[iy]:.2f}, {q0[iz]:.2f})")
qlo[ix], qhi[ix] = -8, -4
qlo[iy], qhi[iy] = -4, 0
qlo[j1], qhi[j1] = -0.01, 0.01

Initial configuration: [ 0.3816452   0.51422852 -0.42443798 -2.0943951   0.62336132  0.52627697
 -0.31709081]
Initial (x,y,z) position: (-4.01, -1.00, 0.01)


In [146]:
# Get current end-effector pose for comparison
plant.SetPositions(plant_context, q0)
q0new = plant.GetPositions(plant_context).copy()
X_WG_current = plant.EvalBodyPoseInWorld(plant_context, gripper_body)
print(f"Current base position: ({q0new[ix]:.2f}, {q0new[iy]:.2f}, {q0new[iz]:.2f})")
print(f"Current gripper position: {X_WG_current.translation()}")

# Define target gripper pose in world frame
target_position = np.array([-4.4, -1.525, 0.95])
target_rpy = RollPitchYaw([-np.pi/12, 0, np.pi/2])
X_WG_grasp = RigidTransform(target_rpy, target_position)
print(f"Target gripper position: {target_position}")

# Define target approach pose in world frame
target_position = np.array([-4.4, -1.525, 1.05])
target_rpy = RollPitchYaw([-np.pi/12, 0, np.pi/2])
X_WG_approach = RigidTransform(target_rpy, target_position)
print(f"Target gripper position: {target_position}")

# Define target place pose in world frame
target_position = np.array([-4.4, -1.9, 0.95])
target_rpy = RollPitchYaw([-np.pi/12, 0, np.pi/2])
X_WG_place = RigidTransform(target_rpy, target_position)
print(f"Target gripper position: {target_position}")

Current base position: (-4.01, -1.00, 0.01)
Current gripper position: [-3.58386774 -0.97227634  0.24189196]
Target gripper position: [-4.4   -1.525  0.95 ]
Target gripper position: [-4.4   -1.525  1.05 ]
Target gripper position: [-4.4  -1.9   0.95]


In [147]:
collision_checker = SceneGraphCollisionChecker(
						model=station,
						robot_model_instances=[station.plant().GetModelInstanceByName("mobile_iiwa")],
						edge_step_size=0.01,
						env_collision_padding=0.01,
						self_collision_padding=0.01,)

INFO:drake:Allocating contexts to support implicit context parallelism 8


In [148]:
q0new = q0.copy()
# q0new[iy] = -1.5
q0new[iz] = 0.03
q0new[3:10] = 0
plant.SetPositions(plant_context, q0new)
collision_checker.CheckConfigCollisionFree(q0new)

True

In [149]:
def solve_ik_for_pose(plant: MultibodyPlant, plant_context: Context, X_WG_target: RigidTransform,
					  theta_bound: float = 0.01 * np.pi, pos_tol: float = 0.02, base_tol: float = 0.2) -> tuple:
	world_frame = plant.world_frame()
	gripper_frame = plant.GetFrameByName("body")

	ik = InverseKinematics(plant, plant_context,)
	ik.AddPositionConstraint(gripper_frame, np.zeros(3), 
							world_frame, X_WG_target.translation() - pos_tol,
							X_WG_target.translation() + pos_tol)
	ik.AddOrientationConstraint(world_frame, X_WG_target.rotation(),
								gripper_frame, RotationMatrix(np.eye(3)), theta_bound)
	ik.AddMinimumDistanceLowerBoundConstraint(0.01)

	# Solve IK problem
	prog = ik.get_mutable_prog()
	q_dec = ik.q()
	# prog.AddBoundingBoxConstraint(q0new[[ix,iy]]-base_tol, q0new[[ix,iy]]+base_tol, q_dec[[ix, iy]])
	prog.AddBoundingBoxConstraint(q0new[iz], q0new[iz]+pos_tol, q_dec[iz:iz+1])
	prog.SetInitialGuess(q_dec, q0new)
	result = Solve(prog)

	if result.is_success():
		q_ik = result.GetSolution(ik.q())
		q_ik[active_hi:] = q0[active_hi:]
		plant.SetPositions(plant_context, q_ik)
		X_WG_achieved = plant.EvalBodyPoseInWorld(plant_context, gripper_body)
		print(f"base position: ({q_ik[ix]:.2f}, {q_ik[iy]:.2f}, {q_ik[iz]:.2f})")
		print(f"Achieved gripper position: {X_WG_achieved.translation()}")
		return q_ik
	else:
		print(f"Infeasible constraints: {result.GetInfeasibleConstraintNames(prog)}")
		return None


In [150]:
q_grasp = solve_ik_for_pose(plant, plant_context, X_WG_grasp)
q_approach = solve_ik_for_pose(plant, plant_context, X_WG_approach)
q_place = solve_ik_for_pose(plant, plant_context, X_WG_place)

base position: (-3.70, -1.51, 0.03)
Achieved gripper position: [-4.41943233 -1.52078279  0.95918995]
base position: (-3.73, -1.49, 0.05)
Achieved gripper position: [-4.38300788 -1.50500049  1.03      ]
base position: (-3.64, -1.88, 0.05)
Achieved gripper position: [-4.38256973 -1.88050192  0.94809456]


In [153]:
plant.SetPositions(plant_context, q0new)

## RRT

In [154]:
class RRTNode:
    __slots__ = ("q", "parent")

    def __init__(self, q, parent):
        self.q = q
        self.parent = parent


class RRTTools:
    def __init__(self, collision_checker, q_lo, q_hi, df_start, df_end, step_size=0.1, goal_threshold=0.2, rng=None):
        self.collision_checker = collision_checker
        self.q_lo = np.array(q_lo, dtype=float)[df_start:df_end]
        self.q_hi = np.array(q_hi, dtype=float)[df_start:df_end]
        self.step_size = float(step_size)
        self.goal_threshold = float(goal_threshold)
        self.rng = np.random.default_rng() if rng is None else rng

        samp_lo = self.q_lo.copy()
        samp_hi = self.q_hi.copy()
        self.df_start, self.df_end = df_start, df_end
    
        mask = ~np.isfinite(samp_lo) | ~np.isfinite(samp_hi)
        if np.any(mask):
            # Fallback range if there are unbounded joints; you can tune this.
            center = 0.0
            span = 1.0
            samp_lo[mask] = center - span
            samp_hi[mask] = center + span
        self.samp_lo = samp_lo
        self.samp_hi = samp_hi

    def steer(self, q_from, q_to):
        dq_active = q_to[self.df_start:self.df_end] - q_from[self.df_start:self.df_end]
        d = np.linalg.norm(dq_active)
        if d <= self.step_size:
            return q_to

        q_new = q_from.copy()
        q_new[self.df_start:self.df_end] = q_from[self.df_start:self.df_end] + (self.step_size / max(d, 1e-12)) * dq_active
        return q_new
        
    def sample_config(self):
        q = np.zeros_like(self.q_lo)
        q[self.df_start:self.df_end] = self.rng.uniform(self.samp_lo[self.df_start:self.df_end],
                                             self.samp_hi[self.df_start:self.df_end])
        q = np.concatenate([self.fixed_head, q, self.fixed_tail])
        return q

    @staticmethod
    def nearest(tree, q):
        qs = np.stack([n.q for n in tree], axis=0)
        return int(np.argmin(np.linalg.norm(qs - q, axis=1)))

    @staticmethod
    def add_node(tree, q, parent_idx):
        tree.append(RRTNode(q, parent_idx))
        return len(tree) - 1

    @staticmethod
    def build_path(tree, idx):
        p = []
        while idx is not None:
            node = tree[idx]
            p.append(node.q)
            idx = node.parent
        return list(reversed(p))

    def connect_greedy(self, tree, q_target):
        checker = self.collision_checker
        step_size = self.step_size

        idx_curr = self.nearest(tree, q_target)
        q_curr = tree[idx_curr].q

        while True:
            d = np.linalg.norm(q_target - q_curr)
            if d < step_size:
                if checker.CheckEdgeCollisionFree(q_curr, q_target):
                    idx_new = self.add_node(tree, q_target, idx_curr)
                    return idx_new, True
                else:
                    return idx_curr, False

            q_next = self.steer(q_curr, q_target)
            if not checker.CheckEdgeCollisionFree(q_curr, q_next):
                return idx_curr, False

            idx_next = self.add_node(tree, q_next, idx_curr)
            idx_curr = idx_next
            q_curr = tree[idx_curr].q

    def plan(self, q_start, q_goal, max_iterations=10000):
        checker = self.collision_checker
        if not checker.CheckConfigCollisionFree(q_start):
            print("[RRT] Start configuration is in collision.")
            return None
        if not checker.CheckConfigCollisionFree(q_goal):
            print("[RRT] Goal configuration is in collision.")
            return None

        T_start = [RRTNode(q_start, None)]
        T_goal  = [RRTNode(q_goal,  None)]
        self.fixed_head = q_start[:self.df_start]
        self.fixed_tail = q_start[self.df_end:]

        for it in range(max_iterations):
            if it % 1000 == 0:
                print(f"[RRT] iteration {it}")
            q_rand = self.sample_config()

            # Alternate which tree grows
            if it % 2 == 0:
                Ta, Tb = T_start, T_goal
            else:
                Ta, Tb = T_goal, T_start

            idx_a_near = self.nearest(Ta, q_rand)
            q_a_near = Ta[idx_a_near].q
            q_a_new = self.steer(q_a_near, q_rand)

            if not checker.CheckEdgeCollisionFree(q_a_near, q_a_new):
                continue

            idx_a_new = self.add_node(Ta, q_a_new, idx_a_near)
            q_a = Ta[idx_a_new].q
            idx_b, complete = self.connect_greedy(Tb, q_a)

            if complete:
                print(f"[RRT] Connected in {it+1} iterations")
                # Compute paths in the original start/goal trees
                path_a = self.build_path(T_start, idx_a_new if Ta is T_start else idx_b)
                path_b = self.build_path(T_goal,  idx_b     if Tb is T_goal  else idx_a_new)
                if Ta is T_goal:
                    path_a, path_b = path_b, path_a
                # path_b is connection→goal; avoid double-counting the connection
                return path_a + path_b[-2::-1]

        print("[RRT] Failed to find a path.")
        return None

    @staticmethod
    def check_equal(a, b, atol: float = 1e-9) -> bool:
        """Robust waypoint equality for arrays / lists."""
        return np.allclose(np.asarray(a), np.asarray(b), atol=atol, rtol=0.0)

    @staticmethod
    def splice_with_shortcut(
        path: list[np.ndarray],
        i: int,
        j: int,
        edge: list[np.ndarray],
    ) -> list[np.ndarray]:
        """
        Replace inclusive subpath path[i:j+1] with 'edge' (path[i]→path[j]).
        """
        prefix = path[:i]
        suffix = path[j+1:]
        return prefix + edge + suffix

    @staticmethod
    def interpolate_edge(q_i: np.ndarray, q_j: np.ndarray, max_step: float,) -> list[np.ndarray]:
        q_i = np.asarray(q_i, dtype=float)
        q_j = np.asarray(q_j, dtype=float)
        d = np.linalg.norm(q_j - q_i)
        if d < 1e-12:
            return [q_i.copy(), q_j.copy()]
        n_steps = max(1, int(np.ceil(d / max_step)))
        ts = np.linspace(0.0, 1.0, n_steps + 1)
        return [(1 - t) * q_i + t * q_j for t in ts]

    def shortcut_path(self, path: list[np.ndarray], passes: int = 200,
                      min_separation: int = 2, max_step: float = 0.1) -> list[np.ndarray]:
        if not path or len(path) < 3:
            return path

        checker = self.collision_checker
        rng = self.rng
        current = [np.asarray(q, dtype=float) for q in path]

        for _ in range(passes):
            n = len(current)
            if n < 3:
                break

            # Choose i < j with some separation
            i = int(rng.integers(0, n - min_separation))
            j = int(rng.integers(i + min_separation, n))
            q_i, q_j = current[i], current[j]

            edge = self.interpolate_edge(q_i, q_j, max_step=max_step)

            # Check the whole edge for collisions
            collision_free = True
            for k in range(len(edge) - 1):
                if not checker.CheckEdgeCollisionFree(edge[k], edge[k+1]):
                    collision_free = False
                    break

            if not collision_free:
                continue

            # Defensive: ensure endpoints match
            if not self.check_equal(edge[0], q_i) or not self.check_equal(edge[-1], q_j):
                continue

            current = self.splice_with_shortcut(current, i, j, edge)

        # Final deduplication
        cleaned = []
        prev = None
        for q in current:
            if prev is None or not self.check_equal(q, prev):
                cleaned.append(q)
            prev = q

        return cleaned

In [ ]:
tools = RRTTools(
        collision_checker=collision_checker,
        q_lo=qlo, q_hi=qhi,
        df_start=active_lo, df_end=active_hi,
        step_size=0.01,
        goal_threshold=0.01,
    )

rrt_path_pick = tools.plan(q_start=q0new, q_goal=q_approach, max_iterations=50_000)

[RRT] iteration 0
[RRT] Connected in 1 iterations


In [157]:
rrt_path_place = tools.plan(q_start=q_approach, q_goal=q_place, max_iterations=50_000)

[RRT] iteration 0
[RRT] Connected in 993 iterations


In [158]:
def visualize_rrt_waypoints(rrt_path, station, meshcat, robot_name="mobile_iiwa", ee_body_name="iiwa_link_ee"):
    plant = station.GetSubsystemByName("plant")
    context = station.CreateDefaultContext()
    plant_context = plant.GetMyMutableContextFromRoot(context)

    robot_instance = plant.GetModelInstanceByName(robot_name)
    ee_body = plant.GetBodyByName(ee_body_name, robot_instance)
    meshcat.Delete("rrt_waypoints")
    sphere = Sphere(0.03)

    for i, q in enumerate(rrt_path):
        q = np.asarray(q).flatten()
        plant.SetPositions(plant_context, q)
        X_WG = plant.EvalBodyPoseInWorld(plant_context, ee_body)

        # Color: green start, red goal, blue intermediates
        if i == 0:
            color = Rgba(0.0, 1.0, 0.0, 0.8)
        elif i == len(rrt_path) - 1:
            color = Rgba(1.0, 0.0, 0.0, 0.8)
        else:
            color = Rgba(0.0, 0.0, 1.0, 0.5)

        path = f"rrt_waypoints/wp_{i:03d}"
        meshcat.SetObject(path, sphere, color)
        meshcat.SetTransform(path, X_WG)

    print(f"visualized {len(rrt_path)} RRT waypoints in Meshcat.")

In [159]:
visualize_rrt_waypoints(rrt_path_place, station, meshcat)

visualized 354 RRT waypoints in Meshcat.


In [160]:
shortcutted_rrt_path_place = tools.shortcut_path(rrt_path_place, passes=200, min_separation=2, max_step=0.1)

In [ ]:
dt = 0.05
pause_before_action = 0.5
pause_after_action = 0.5
opened, closed = 0.107, 0.0

def _append_path(times: list[float], Q: list[np.ndarray], t: float, path: list[np.ndarray]) -> float:
    for q in path:
        if times: t += dt
        times.append(t)
        Q.append(q)
    return t

def _hold(times: list[float], Q: list[np.ndarray], t: float,
    q_hold: np.ndarray, duration: float,) -> float:
    if duration <= 0: return t
    t += duration
    times.append(t)
    Q.append(q_hold)
    return t

def build_trajs(
    path_pick: list[np.ndarray],
    path_place: list[np.ndarray],
    q_grasp: np.ndarray,
    q_approach: np.ndarray,
    # path_place: list[np.ndarray],
    # path_reset: list[np.ndarray],
) -> tuple["PiecewisePolynomial", "PiecewisePolynomial"]:
    """Sequence:
    pick → q_grasp (OPEN) → pause 0.5 → CLOSE (no motion) → pause 0.5 → q_approach →
    place → pause 0.5 → OPEN (no motion) → pause 0.5 → reset
    """
    times: list[float] = []
    Q: list[np.ndarray] = []
    t = 0.0

    # 1) path_pick  (ends at q_approach)
    t = _append_path(times, Q, t, path_pick)

    # 2) move to q_grasp (WSG stays OPEN)
    if not np.allclose(Q[-1], q_grasp):
        t += 10 * dt
        times.append(t)
        Q.append(q_grasp)

    # 3) pause BEFORE CLOSE
    t = _hold(times, Q, t, q_grasp, pause_before_action)

    # 4) CLOSE
    t_close = t
    t = _hold(times, Q, t, q_grasp, pause_after_action)
    
    # 5) return to q_approach
    if not np.allclose(Q[-1], q_approach):
        t += 10 * dt
        times.append(t)
        Q.append(q_approach)

    # 6) path_place (ends at q_place)
    t = _append_path(times, Q, t, path_place)

    # 7) pause BEFORE OPEN (no motion)v
    t = _hold(times, Q, t, Q[-1], pause_before_action)

     # 8) OPEN (command change at this instant), then pause AFTER OPEN (no motion)
    t_open = t
    t = _hold(times, Q, t, Q[-1], pause_after_action)



    q_samples = np.stack(Q, axis=1)
    print(q_samples.shape)
    traj_q = PiecewisePolynomial.FirstOrderHold(times, q_samples)

    wsg_knots = [times[0], t_close, times[-1]]
    wsg_vals = [opened, closed, closed]
    traj_wsg = PiecewisePolynomial.ZeroOrderHold(
        wsg_knots, np.asarray(wsg_vals).reshape(1, -1)
    )

    print(f"T={times[-1]:.3f}s")
    return traj_q, traj_wsg

In [134]:
shortcutted_rrt_path = [cut_path[:10] for cut_path in shortcutted_rrt_path]
traj_q, traj_wsg = build_trajs(shortcutted_rrt_path, q_grasp[:10], q_approach[:10])

(10, 19)
T=2.700s


In [136]:
model_drivers = """
model_drivers:
  wsg: !SchunkWsgDriver {}
  mobile_iiwa: !InverseDynamicsDriver {}
"""

scenario_data_w_driver = scenario_data + model_drivers

In [ ]:
class IiwaDesiredStateFromQ(LeafSystem):
    def __init__(self, nq, nv):
        super().__init__()
        self.nq = nq
        self.nv = nv
        self.DeclareVectorInputPort("q_des", BasicVector(nq))
        self.DeclareVectorOutputPort(
            "x_des", BasicVector(nq + nv), self.CalcOutput
        )

    def CalcOutput(self, context: Context, output: BasicVector):
        q_des = self.get_input_port(0).Eval(context)
        x_des = np.zeros(self.nq + self.nv)
        x_des[:self.nq] = q_des
        output.SetFromVector(x_des)


scenario = LoadScenario(data=scenario_data_w_driver)
builder = DiagramBuilder()
station = builder.AddSystem(MakeHardwareStation(scenario, meshcat=meshcat))

plant = station.GetSubsystemByName("plant")
mobile_iiwa = plant.GetModelInstanceByName("mobile_iiwa")

nq = plant.num_positions(mobile_iiwa)
nv = plant.num_velocities(mobile_iiwa)

# Trajectory sources
traj_source_q = builder.AddSystem(TrajectorySource(traj_q))
traj_source_wsg = builder.AddSystem(TrajectorySource(traj_wsg))
q_to_x = builder.AddSystem(IiwaDesiredStateFromQ(nq, nv))

builder.Connect(
    traj_source_q.get_output_port(),
    q_to_x.get_input_port(0),
)

# Feed desired state into the InverseDynamicsDriver
builder.Connect(
    q_to_x.get_output_port(),
    station.GetInputPort("mobile_iiwa.desired_state"),
)

builder.Connect(
    traj_source_wsg.get_output_port(),
    station.GetInputPort("wsg.position"),
)

diagram = builder.Build()
simulation = Simulator(diagram)
simulation.set_target_realtime_rate(1.0)

ctx = simulation.get_mutable_context()
plant_context = plant.GetMyMutableContextFromRoot(ctx)

# Make the physical plant start at first waypoint of traj_q
q0 = traj_q.value(0.0).flatten()
plant.SetPositions(plant_context, mobile_iiwa, q0)
plant.SetVelocities(plant_context, mobile_iiwa, np.zeros(nv))

diagram.ForcedPublish(ctx)
meshcat.StartRecording()
simulation.Initialize()
simulation.AdvanceTo(max(traj_q.end_time(), traj_wsg.end_time()))
meshcat.StopRecording()
meshcat.PublishRecording()